# Rewrites the outputted ONNX file to only output the positive probability

In [ ]:
import onnx
import numpy as np
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
import Utils
import matplotlib.pyplot as plt
import importlib
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
## Code to use gather to select positive class score from probabilities output rewrites model


# Load model
model = onnx.load("model_all_features.onnx")
graph = model.graph

# ---- Step 1: find probabilities output ----
# Usually output[1], but safer to check names
prob_output = graph.output[1].name

# ---- Step 2: create Gather node to select index 1 ----
# We use Gather(axis=1) to pick the positive class

# Constant tensor: index = 1
index_name = "pos_index"
index_tensor = numpy_helper.from_array(
    np.array([1], dtype=np.int64),
    name=index_name
)

gather_node = helper.make_node(
    "Gather",
    inputs=[prob_output, index_name],
    outputs=["positive_score"],
    axis=1
)

# ---- Step 3: add node + initializer ----
graph.node.append(gather_node)
graph.initializer.append(index_tensor)

# ---- Step 4: replace outputs ----
graph.output.clear()

graph.output.append(
    helper.make_tensor_value_info(
        "positive_score",
        TensorProto.FLOAT,
        [None, 1]
    )
)

# ---- Step 5: (optional but smart) remove label output ----
# Not strictly required, but avoids confusion
# You could also leave it if downstream expects 2 outputs
onnx.save(model, "model_all_features_positive.onnx")